# 2 · Conjuring geometry & taming the mesh 🍫

:::{dropdown} ▶ How to run / view this notebook
:class: howto-run

| 💻 Local | 🌐 Static | ▶ JupyterLite | ☁️ Colab |
|:--:|:--:|:--:|:--:|
| ✅ | [✅](/) | [⌛ ~1 min](/lite/notebooks/index.html?path=02-geometry.ipynb) | [✅](https://colab.research.google.com/github/schruste/ngsum2026-colab/blob/colab/02-geometry.ipynb) |

<sub>✅ runs here · ⌛ runs but slowly (rough time). Use the ⚙ **View options** to
switch story / quizzes / gimmicks on or off.</sub>
:::

:::{dropdown} 🎭 Story — feeding the Beast
:class: storytelling

*The masters have withdrawn; now you must approach the Beast alone. No adventurer
commands it without first commanding the **ground** it stands on — and a wary Beast is
best met with a gift. So you conjure its favourite snack, a **chocolate bar** 🍫, and learn
the craft of **geometry and mesh** in the doing. (The Beast is much more agreeable on a
full stomach.)*
:::

This unit is about **geometry & meshing**: conjuring a solid with NGSolve's CAD kernel
(Netgen/OpenCASCADE), the **knobs** that control a mesh, **querying mesh topology**,
**evaluating functions at points**, and **importing a real CAD part**. We keep each a
short highlight and point to the references for depth.

In [ ]:
# --- Google Colab: install NGSolve on first run (a no-op anywhere else) -------
# NGSolve ships its PyPI wheels as pre-releases, so the `--pre` flag is essential.
import sys
if "google.colab" in sys.modules:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "--pre",
                    "ngsolve", "webgui_jupyter_widgets"], check=True)

In [ ]:
from netgen.occ import (Box, Prism, Face, Wire, Segment, Pnt, Vec, OCCGeometry,
                        WorkPlane, Axes, Axis, X, Y, Z, Sphere, Cylinder, Glue)
from ngsolve import Mesh, H1
from ngsolve.meshes import Make1DMesh
from ngsolve.webgui import Draw
import matplotlib.pyplot as plt

view = dict(euler_angles=[-70, 0, -20])              # a nice 3/4 view

## 1. Conjure a solid from nothing — a chocolate bar

The OCC workflow is almost always **points → edges/wire → face → solid**. We
sketch a trapezoid and pull it along an axis with `Prism` to get a *ridge*
("roof"); the **intersection** (`*`) of two perpendicular ridges is a peak. A few
peaks on a base, rounded valleys (`MakeFillet`) and a flat cut (`*` with a box):
a bar conjured from pure parameters.

In [ ]:
n_peaks, b, depth, base_h, ph, r_fil = 3, 2.0, 3.0, 0.4, 2.6, 0.2
W = n_peaks * b

def trapezoid(a, b, c, d):
    return Face(Wire([Segment(a, b), Segment(b, c), Segment(c, d), Segment(d, a)]))

def peak(cx, cy, z0, bx, by, h):
    """A trapezoidal peak: the intersection of two perpendicular ridges."""
    rx = Prism(trapezoid(Pnt(cx-bx/2, cy-by/2, z0), Pnt(cx+bx/2, cy-by/2, z0),
                         Pnt(cx+0.1*bx, cy-by/2, z0+h), Pnt(cx-0.1*bx, cy-by/2, z0+h)),
               Vec(0, by, 0))
    ry = Prism(trapezoid(Pnt(cx-bx/2, cy-by/2, z0), Pnt(cx-bx/2, cy+by/2, z0),
                         Pnt(cx-bx/2, cy+0.1*by, z0+h), Pnt(cx-bx/2, cy-0.1*by, z0+h)),
               Vec(bx, 0, 0))
    return rx * ry

m = 0.3                                              # build over-long, then cut flat
bar = Box(Pnt(0, -m, 0), Pnt(W, depth + m, base_h))
for i in range(n_peaks):
    bar = bar + peak((i + 0.5) * b, depth/2, base_h, b, depth + 2*m, ph)

def along_y(e):
    p, q = (v.p for v in e.vertices)
    return abs(p[1] - q[1]) > abs(p[0] - q[0])
valleys = [e for e in bar.edges if abs(e.center[2] - base_h) < 0.05
           and along_y(e) and 0.1 < e.center[0] < W - 0.1]
bar = bar.MakeFillet(valleys, r_fil)                       # round the valleys
bar = bar * Box(Pnt(-1, 0, -1), Pnt(W + 1, depth, 10*ph))  # cut front/back flat
bar.faces.col = (0.42, 0.26, 0.10)
print(f"a chocolate bar, conjured: {len(bar.faces)} faces, volume {bar.mass:.1f}")
Draw(bar, **view)

:::{dropdown} 🎭 Story — a peace offering
:class: storytelling

*You hold out the freshly conjured chocolate bar. The Beast, until now all wild edges and
suspicion, leans in and sniffs — and softens. It lets you rest a hand on its hide. The
first trust is won, paid for in chocolate.*

```{image} data/beast-tamed.jpg
:alt: The pirate offers the chocolate bar to the now-calm Beast — the first trust is won
:width: 360px
:align: center
```
:::

## 2. From solid to mesh — and the knobs that matter

A geometry is not yet a *mesh*. `GenerateMesh(maxh=...)` sets the (maximum)
element size — smaller means more, smaller elements: more accurate, more
expensive. The OCC workflow **sketch → face → solid** also lives in 2D (a
`WorkPlane` is the whole sketch pad) and the boundary can be **curved**.

We meet the cup ☕ — a companion for several notebooks — both as a **3D body of
revolution** and as its **2D cross-section**.

In [ ]:
def cup_profile():
    """Half cross-section of a cup (x = radius), sketched in the x–z plane."""
    R, wall, base, Hc = 2.4, 0.4, 0.6, 6.0
    return (WorkPlane(Axes((0, 0, 0), n=Y, h=X))
            .MoveTo(0, 0).LineTo(R, 0).LineTo(R, Hc).LineTo(R - wall, Hc)
            .LineTo(R - wall, base).LineTo(0, base).Close().Face())

mug = cup_profile().Revolve(Axis(Pnt(0, 0, 0), Z), 360)    # spin the profile 360°
mesh3 = Mesh(OCCGeometry(mug).GenerateMesh(maxh=1.0))
print(f"3D mug: {mesh3.ne} volume elements, {mesh3.nv} vertices")

**The Beast won't behave… until we feed it.** Straight-sided elements approximate
the round wall by a crude **polygon** — the hide is *jagged*, and nothing built on
it will be accurate. Feed the Beast the **chocolate bar**, and `mesh.Curve(order)`
**bends the elements** to follow the true geometry — suddenly it is smooth and
obedient. (The same `Curve` is what makes high-order methods worthwhile later.)

In [ ]:
Draw(mesh3)                                          # jagged: the polygonal hide
mesh3.Curve(3)                                       # 🍫 fed — now it follows the curve
Draw(mesh3)                                          # smooth: the Beast obeys

**Mesh size & local refinement (2D).** In 2D the cross-section is a flat sketch.
`maxh` controls the global size; setting `.maxh` on *one* shape refines only
there — here the thin handle, while the body stays coarse.

In [ ]:
def coffee_cup(h_handle=None):
    """A 2D coffee-cup cross-section: trapezoidal body + a ring handle."""
    Wb, Wt, H = 4.0, 5.0, 6.0
    body = (WorkPlane().MoveTo(-Wb/2, 0).LineTo(Wb/2, 0)
            .LineTo(Wt/2, H).LineTo(-Wt/2, H).Close().Face())
    cx, cy = Wt/2 + 0.2, H/2
    outer = WorkPlane(Axes((cx, cy, 0), n=Z)).Circle(0, 0, 1.7).Face()
    inner = WorkPlane(Axes((cx, cy, 0), n=Z)).Circle(0, 0, 1.0).Face()
    handle = outer - inner
    if h_handle:
        handle.maxh = h_handle                       # local mesh size on the handle only
    return body + handle

for h in [1.2, 0.6, 0.3]:
    m2 = Mesh(OCCGeometry(coffee_cup(), dim=2).GenerateMesh(maxh=h))
    print(f"maxh = {h}:  {m2.ne:5d} triangles,  {m2.nv:5d} vertices")
mesh2 = Mesh(OCCGeometry(coffee_cup(h_handle=0.2), dim=2).GenerateMesh(maxh=0.8))
mesh2.Curve(4)
print("fine handle, coarse body:", mesh2.ne, "triangles")
for kk in [1, 2, 3]:
    print(f"H1 order {kk}:  {H1(mesh2, order=kk).ndof} degrees of freedom")
Draw(mesh2)

**1D too.** A mesh can be one-dimensional — a chain of intervals. `Make1DMesh(n)`
gives `n` equal cells on $[0,1]$; a `mapping` **grades** them (clustering where the
solution is steep, e.g. a boundary layer).

In [ ]:
def nodes(m): return sorted(p[0] for p in m.ngmesh.Points())
uniform = Make1DMesh(10)
graded = Make1DMesh(10, mapping=lambda t: t**1.6)
plt.figure(figsize=(6, 1.5))
plt.plot(nodes(uniform), [1]*uniform.nv, "o-", label="uniform")
plt.plot(nodes(graded),  [0]*graded.nv,  "s-", label="graded")
plt.yticks([0, 1], ["graded", "uniform"]); plt.ylim(-0.5, 1.5); plt.legend(loc="center right")
plt.tight_layout(); plt.show()

## 3. Knowing your mesh — topology & evaluating at a point

A mesh is also a **queryable object**. Ask it how many vertices and elements it has,
which **materials** (subdomains) and **boundary regions** it carries, or iterate over
elements and their vertices — the topology the assembly later walks. And any
`CoefficientFunction` (next unit) can be **evaluated at a world point**: `mesh(x, y)`
maps that point into its element, and calling the function on it returns the value.

In [ ]:
from ngsolve import ElementId, VOL, x, y, CF
print("mesh2:", mesh2.nv, "vertices,", mesh2.ne, "elements")
print("materials (subdomains):", mesh2.GetMaterials())
print("boundary regions:   ", mesh2.GetBoundaries())
el0 = mesh2[ElementId(VOL, 0)]                       # the first element …
print("element 0 has vertices", [v.nr for v in el0.vertices])
g = CF(x * y)                                        # any CoefficientFunction (unit 3)
print("x*y at the world point (0, 2):", g(mesh2(0.0, 2.0)))

## 4. Importing a real CAD part for the Beast

In the wild, geometry rarely comes hand-sketched — it arrives as a **CAD file**
(`STEP`, `IGES`, `BREP`) from FreeCAD, SolidWorks, … NGSolve **imports** it, lets
you **modify** it, and meshes it like any other shape. We take a genuine external
part — a **threaded screw** (`data/screw.step`) — and bolt it onto **the Beast**
(the *sculpture* mascot: a spherical shell bored by three cylinders).

In [ ]:
screw = OCCGeometry("data/screw.step").shape
b0, b1 = screw.bounding_box
cx, cy = (b0[0] + b1[0]) / 2, (b0[1] + b1[1]) / 2
screw = screw.Move((-cx, -cy, -b0[2])).Scale(Pnt(0, 0, 0), 1.6)        # recentre + enlarge
screw.faces.col = (0.72, 0.72, 0.75)
screw.maxh = 3.0                                     # the fine thread needs a local mesh size
Draw(screw)

def beast_sculpture():
    s = Sphere(Pnt(50, 50, 50), 80) - Sphere(Pnt(50, 50, 50), 50)
    for p, d in [(Pnt(-100, 0, 0), X), (Pnt(100, -100, 100), Y), (Pnt(0, 100, -100), Z)]:
        s = s - Cylinder(p, d, r=40, h=300)
    return s.Move((-50, -50, -50))

beast = beast_sculpture()
beast.faces.col = (0.55, 0.45, 0.30)
top = beast.bounding_box[1][2]
screw = screw.Move((0, 0, top - 18))                 # drive the shaft into the crown
beast.solids.name = "beast"; screw.solids.name = "screw"
asm = Glue([beast, screw])                           # one body, two named parts
mesh = Mesh(OCCGeometry(asm).GenerateMesh(maxh=20)); mesh.Curve(2)
print(f"Beast + bolt: {mesh.ne} elements; parts: {set(mesh.GetMaterials())}")
clip = {"Clipping": {"enable": True, "function": True, "x": 0, "y": 1, "z": 0, "dist": 0}}
Draw(mesh, settings=clip)

:::{dropdown} 📚 Further reading
:class: further-reading

- **Constructive solid geometry** — i-tutorial
  [4.2 CSG](https://docu.ngsolve.org/latest/i-tutorials/unit-4.2-csg/csg.html).
- **OpenCASCADE (OCC) geometry & CAD import** — i-tutorial
  [4.4 OCC geometry](https://docu.ngsolve.org/latest/i-tutorials/unit-4.4-occ/occ.html).
- **Spaces & forms on subdomains** — i-tutorial
  [1.5 subdomains](https://docu.ngsolve.org/latest/i-tutorials/unit-1.5-subdomains/subdomains.html).
:::

:::{dropdown} 🧠 Quiz — why did the thread need its own `maxh`, and what does `Glue` do?
:class: quiz
CAD parts carry **fine features** (the screw's thread) far smaller than the body
they sit on; a coarse global size makes the mesher choke, so we pin a **local**
`screw.maxh`. `Glue` welds two solids into one body sharing an **internal
interface** (continuous fields across it) while keeping them as **separate named
parts** — exactly what we need when materials differ. (Use `+` for one merged
solid.) And meshing a curved CAD surface, like the cup, still wants `mesh.Curve`.
:::

You can now conjure terrain, mesh it, curve it and import it. Next the Beast asks a
sharper question: *what, exactly, is a function on this mesh?* — the
**CoefficientFunction**.

In [ ]:
# Navigation to the next unit — shown only in a live notebook (Colab /
# JupyterLite / local Jupyter), never in the rendered website.
import os, sys
if not os.environ.get("WEBGUI_SCENE_DIR"):          # not the static site build
    _nb, _title = "03-coefficientfunctions", "3 · What is a CoefficientFunction? 🔨"
    if "google.colab" in sys.modules:
        _u = "https://colab.research.google.com/github/schruste/ngsum2026-colab/blob/colab/" + _nb + ".ipynb"
    else:                                           # JupyterLite & local open relative .ipynb links
        _u = _nb + ".ipynb"
    from IPython.display import display, Markdown
    display(Markdown("➡️ **Next unit:** [" + _title + "](" + _u + ")"))